# Hybrid Search & Reranking in FFAI's RAG Module

This notebook demonstrates the search subsystem in FFAI's RAG module using
the **real Mistral Small API** via `FFLiteLLMClient` for query expansion
and real Mistral embeddings for vector search.

We cover:

1. BM25 keyword search
2. Vector search with Mistral embeddings
3. Hybrid search with Reciprocal Rank Fusion (RRF)
4. Standalone RRF for multi-list merging
5. Reranking (Noop / Diversity / CrossEncoder / Factory)
6. Query expansion
7. End-to-end search pipeline
8. Summary table

<div class="page-break"></div>

---

## Section 1: Setup

In [ ]:
import os
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from dotenv import load_dotenv  # noqa: E402

load_dotenv()

api_key = os.getenv('MISTRAL_API_KEY')
if not api_key:
    raise RuntimeError('Set MISTRAL_API_KEY in your environment or .env file')

print(f'Project root: {_project_root}')

In [ ]:
from ffai.Clients.FFLiteLLMClient import FFLiteLLMClient
from ffai.rag.embed import Embeddings
from ffai.rag.indexing import BM25Index
from ffai.rag.search import (
    CrossEncoderReranker,
    DiversityReranker,
    HybridSearch,
    NoopReranker,
    QueryExpander,
    fuse_search_results,
    get_reranker,
    reciprocal_rank_fusion,
)

client = FFLiteLLMClient(
    model_string='mistral/mistral-small-latest',
    api_key=api_key,
    temperature=0.3,
    max_tokens=256,
)

embed = Embeddings(model='mistral/mistral-embed', api_key=api_key, cache_enabled=True)

print('All imports successful')

<div class="page-break"></div>

---

## Section 2: BM25 Keyword Search

BM25 (Okapi BM25) is a bag-of-words ranking function that scores
documents based on term frequency, inverse document frequency, and
document length normalization. It excels at exact keyword matching.

We index 10 documents covering different programming topics
(Python async, Rust ownership, Go goroutines, JavaScript promises, etc.).

In [ ]:
bm25 = BM25Index()

docs = [
    ("python_async", "Python asyncio provides asynchronous programming with async/await syntax. "
     "Coroutines and event loops enable concurrent I/O operations without threads."),
    ("rust_ownership", "Rust ownership model ensures memory safety without garbage collection. "
     "The borrow checker enforces strict rules about variable lifetimes and references."),
    ("go_goroutines", "Go goroutines are lightweight concurrency primitives managed by the Go runtime. "
     "Channels provide safe communication between goroutines for concurrent programming."),
    ("js_promises", "JavaScript promises handle asynchronous operations in single-threaded environments. "
     "Async/await syntax simplifies promise chains for concurrent task management."),
    ("java_threads", "Java threads enable concurrent programming with shared memory. "
     "Synchronized blocks and locks prevent race conditions in multi-threaded code."),
    ("cpp_memory", "C++ smart pointers provide memory safety through RAII patterns. "
     "Unique pointers enforce ownership while shared pointers use reference counting."),
    ("rust_concurrency", "Rust concurrency combines ownership with threads for safe parallel execution. "
     "The type system prevents data races at compile time through Send and Sync traits."),
    ("elixir_actor", "Elixir uses the actor model for concurrent programming via lightweight processes. "
     "Message passing between processes eliminates shared-state concurrency bugs."),
    ("python_error", "Python exception handling uses try/except/finally blocks for error management. "
     "Custom exception classes enable structured error propagation in async code."),
    ("kotlin_coroutines", "Kotlin coroutines simplify asynchronous programming on the JVM. "
     "Structured concurrency ensures all launched coroutines complete predictably."),
]

for doc_id, content in docs:
    bm25.add_document(doc_id, content)

print(f'Indexed {bm25.count()} documents')
print(f'Index stats: {bm25.get_stats()}')

In [ ]:
results = bm25.search('concurrent programming', n_results=5)
print('Query: "concurrent programming"')
print('-' * 70)
for r in results:
    print(f"  {r['id']:25s}  score={r['score']:.4f}")
    print(f"    {r['content'][:90]}...")
print()

In [ ]:
results = bm25.search('memory safety', n_results=5)
print('Query: "memory safety"')
print('-' * 70)
for r in results:
    print(f"  {r['id']:25s}  score={r['score']:.4f}")
    print(f"    {r['content'][:90]}...")
print()

In [ ]:
results_async = bm25.search('async error handling', n_results=5)
print('Query: "async error handling"')
print('-' * 70)
for r in results_async:
    print(f"  {r['id']:25s}  score={r['score']:.4f}")
print()
print('Notice: docs containing multiple query terms score higher')
print('(python_async matches "async", python_error matches "error" & "handling")')

<div class="page-break"></div>

---

## Section 3: Vector Search with Mistral Embeddings

Vector search compares semantic meaning via embedding similarity.
We use the real Mistral embeddings API to compute document and query vectors.

In [ ]:
doc_contents = dict(docs)
doc_texts = [content for _, content in docs]
doc_ids = [doc_id for doc_id, _ in docs]
doc_vectors = embed.embed(doc_texts)

print(f'Embedded {len(doc_vectors)} documents, dimension={len(doc_vectors[0])}')

In [ ]:
def vector_search_fn(query: str, n_results: int) -> list[dict]:
    query_vec = embed.embed_single(query)
    scored = []
    for i, doc_vec in enumerate(doc_vectors):
        sim = Embeddings.cosine_similarity(query_vec, doc_vec)
        scored.append({
            'id': doc_ids[i],
            'content': doc_contents[doc_ids[i]],
            'score': sim,
            'metadata': {},
        })
    scored.sort(key=lambda x: x['score'], reverse=True)
    return scored[:n_results]


vec_results = vector_search_fn('concurrent programming', 5)
print('Vector search: "concurrent programming"')
print('-' * 70)
for r in vec_results:
    print(f"  {r['id']:25s}  score={r['score']:.6f}")
    print(f"    {r['content'][:90]}...")

<div class="page-break"></div>

---

## Section 4: Hybrid Search with Reciprocal Rank Fusion

Hybrid search combines BM25 keyword matching and vector semantic
similarity via **Reciprocal Rank Fusion** (RRF). The `alpha` parameter
controls the weight: `alpha` favors vector search, `1-alpha` favors BM25.

RRF score = `alpha/(k + vector_rank) + (1-alpha)/(k + bm25_rank)`.

In [ ]:
def bm25_search_fn(query: str, n_results: int) -> list[dict]:
    return bm25.search(query, n_results=n_results)


hybrid = HybridSearch(
    vector_search_fn=vector_search_fn,
    bm25_search_fn=bm25_search_fn,
    alpha=0.6,
    rrf_k=60,
)

results = hybrid.search('error handling in async code', n_results=5)
print('Hybrid search: "error handling in async code" (alpha=0.6)')
print('-' * 90)
print(f"  {'ID':25s}  {'vec_score':>10s}  {'bm25_score':>10s}  {'rrf_score':>10s}")
print('-' * 90)
for r in results:
    vs = r.get('vector_score', 0)
    bs = r.get('bm25_score', 0)
    rrf = r.get('rrf_score', 0)
    print(f"  {r['id']:25s}  {vs:10.6f}  {bs:10.4f}  {rrf:10.6f}")

In [ ]:
print('Effect of alpha on hybrid ranking for: "error handling in async code"')
print('=' * 90)
for alpha_val in [0.9, 0.6, 0.3]:
    hybrid.set_alpha(alpha_val)
    results = hybrid.search('error handling in async code', n_results=3)
    print(f'\nalpha={alpha_val:.1f} (vector weight={alpha_val:.0%}, bm25 weight={1-alpha_val:.0%}):')
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r['id']:25s}  rrf_score={r['rrf_score']:.6f}")

hybrid.set_alpha(0.6)

<div class="page-break"></div>

---

## Section 5: Reciprocal Rank Fusion (Standalone)

The standalone `reciprocal_rank_fusion()` function merges **arbitrary** result
lists -- not just vector+BM25. RRF scores each document as
`sum(weight / (k + rank))` across all lists, so documents appearing near
the top of multiple lists get the highest combined score.

In [ ]:
list_a = [
    {'id': 'python_async', 'content': 'Python asyncio ...'},
    {'id': 'rust_concurrency', 'content': 'Rust concurrency ...'},
    {'id': 'go_goroutines', 'content': 'Go goroutines ...'},
]

list_b = [
    {'id': 'go_goroutines', 'content': 'Go goroutines ...'},
    {'id': 'elixir_actor', 'content': 'Elixir actor model ...'},
    {'id': 'python_async', 'content': 'Python asyncio ...'},
]

list_c = [
    {'id': 'kotlin_coroutines', 'content': 'Kotlin coroutines ...'},
    {'id': 'python_async', 'content': 'Python asyncio ...'},
    {'id': 'rust_concurrency', 'content': 'Rust concurrency ...'},
]

fused = reciprocal_rank_fusion([list_a, list_b, list_c], k=60)
print('Fused results from 3 lists (k=60):')
print('-' * 50)
for r in fused:
    print(f"  {r['id']:25s}  rrf_score={r['rrf_score']:.6f}")

print()
print('"python_async" appears in all 3 lists at various ranks -> highest RRF score')
print('"kotlin_coroutines" appears in only 1 list -> lowest RRF score')

<div class="page-break"></div>

---

## Section 6: Reranking

Rerankers take an initial result set and reorder it using a second-pass
scoring strategy. FFAI ships three rerankers:

- **NoopReranker**: pass-through (no changes)
- **DiversityReranker**: promotes varied results via MMR-style selection
- **CrossEncoderReranker**: neural cross-encoder scoring

In [ ]:
noop = NoopReranker()
sample = [
    {'id': 'a', 'content': 'doc a', 'score': 0.9},
    {'id': 'b', 'content': 'doc b', 'score': 0.7},
    {'id': 'c', 'content': 'doc c', 'score': 0.5},
]
reranked = noop.rerank('test query', sample)
print('NoopReranker -- pass-through (no reordering):')
for r in reranked:
    print(f"  {r['id']}  score={r['score']}")
print(f'Results unchanged: {reranked == sample}')

In [ ]:
overlap_results = [
    {'id': 'r1', 'content': 'Python async await coroutine concurrent task', 'score': 0.95},
    {'id': 'r2', 'content': 'Python async await coroutine event loop', 'score': 0.90},
    {'id': 'r3', 'content': 'Python async await future callback', 'score': 0.85},
    {'id': 'r4', 'content': 'Rust ownership borrow checker lifetime', 'score': 0.70},
    {'id': 'r5', 'content': 'Go goroutine channel message passing', 'score': 0.60},
]

print('BEFORE DiversityReranker (lambda=0.7):')
for i, r in enumerate(overlap_results, 1):
    print(f"  {i}. {r['id']}  score={r['score']:.2f}  {r['content'][:50]}")

div_reranker = DiversityReranker(lambda_param=0.7)
diverse = div_reranker.rerank('async programming', overlap_results)

print('\nAFTER DiversityReranker (lambda=0.7):')
for i, r in enumerate(diverse, 1):
    print(f"  {i}. {r['id']}  diversity_rank={r['diversity_rank']}  score={r['score']:.2f}  {r['content'][:50]}")

print('\nTop result stays (highest relevance), then diverse docs promoted over similar ones')

In [ ]:
ce_reranker = CrossEncoderReranker(model_name='cross-encoder/ms-marco-MiniLM-L-6-v2')

ce_input = [
    {'id': 'd1', 'content': 'async await in Python', 'score': 0.8},
    {'id': 'd2', 'content': 'Rust borrow checker rules', 'score': 0.6},
    {'id': 'd3', 'content': 'Go channels and goroutines', 'score': 0.5},
]

ce_results = ce_reranker.rerank('async programming', ce_input)
print('CrossEncoderReranker results:')
for r in ce_results:
    print(f"  {r['id']}  rerank_score={r['rerank_score']:.4f}  original_score={r['original_score']}")

In [ ]:
for rtype in ('none', 'diversity', 'cross_encoder'):
    r = get_reranker(rtype)
    print(f"  get_reranker('{rtype}') -> {type(r).__name__}")

<div class="page-break"></div>

---

## Section 7: Query Expansion

`QueryExpander` uses the LLM to reformulate a query into multiple
variations, improving recall by catching different phrasings.
We use the real Mistral Small API for expansion.

In [ ]:
def llm_generate_fn(prompt: str) -> str:
    result = client.generate_response(prompt)
    return result


expander = QueryExpander(llm_generate_fn=llm_generate_fn, n_variations=3)
expanded = expander.expand('authentication methods')
print('Original query: "authentication methods"')
print(f'Expanded to {len(expanded)} queries:')
for i, q in enumerate(expanded):
    print(f'  {i + 1}. {q}')

In [ ]:
results_q1 = [
    {'id': 'auth_1', 'content': 'OAuth 2.0 flow', 'score': 0.95},
    {'id': 'auth_2', 'content': 'JWT tokens', 'score': 0.80},
]
results_q2 = [
    {'id': 'auth_3', 'content': 'SAML SSO integration', 'score': 0.90},
    {'id': 'auth_1', 'content': 'OAuth 2.0 flow', 'score': 0.85},
]
results_q3 = [
    {'id': 'auth_4', 'content': 'API key management', 'score': 0.75},
    {'id': 'auth_2', 'content': 'JWT tokens', 'score': 0.70},
]

fused = fuse_search_results([results_q1, results_q2, results_q3], n_results=5)
print('Fused results from 3 expanded queries (deduplicated):')
for r in fused:
    print(f"  {r['id']}  score={r['score']:.2f}  {r['content']}")
print(f'\n{len(fused)} unique results (auth_1 and auth_2 duplicates removed)')

<div class="page-break"></div>

---

## Section 8: End-to-End Search Pipeline

Put it all together: BM25 index -> Hybrid search -> Reranking.

In [ ]:
query = 'safe concurrent error handling'

hybrid.set_alpha(0.5)
stage1 = hybrid.search(query, n_results=8, mode='hybrid')
print(f'Stage 1 -- Hybrid search for "{query}"')
print('-' * 70)
for i, r in enumerate(stage1, 1):
    print(f"  {i}. {r['id']:25s}  rrf={r['rrf_score']:.6f}")

In [ ]:
stage2 = div_reranker.rerank(query, stage1, n_results=5)
print('Stage 2 -- After DiversityReranker (lambda=0.7)')
print('-' * 70)
for i, r in enumerate(stage2, 1):
    print(f"  {i}. {r['id']:25s}  diversity_rank={r['diversity_rank']}  rrf={r.get('rrf_score', 0):.6f}")
    print(f"     {r['content'][:80]}...")

<div class="page-break"></div>

---

## Section 9: Summary

In [ ]:
import polars as pl

summary = pl.DataFrame([
    {'Component': 'BM25Index', 'Purpose': 'Keyword search via Okapi BM25', 'Key Parameters': 'k1, b, epsilon'},
    {'Component': 'HybridSearch', 'Purpose': 'Combine vector + BM25 via RRF', 'Key Parameters': 'alpha, rrf_k, mode'},
    {'Component': 'reciprocal_rank_fusion', 'Purpose': 'Merge arbitrary result lists', 'Key Parameters': 'k, weights'},
    {'Component': 'NoopReranker', 'Purpose': 'Pass-through (disabled reranking)', 'Key Parameters': '\u2014'},
    {'Component': 'DiversityReranker', 'Purpose': 'Promote diverse results via MMR', 'Key Parameters': 'lambda_param'},
    {'Component': 'CrossEncoderReranker', 'Purpose': 'Cross-encoder neural reranking', 'Key Parameters': 'model_name, max_length'},
    {'Component': 'QueryExpander', 'Purpose': 'Multi-query retrieval via LLM', 'Key Parameters': 'n_variations, include_original'},
    {'Component': 'fuse_search_results', 'Purpose': 'Deduplicate merged results', 'Key Parameters': 'n_results, dedupe_by'},
])

print(summary)